# Military Service and Immigrants' Integration: Evidence from the Vietnam Draft Lotteries

**Authors:** Nan Zhang, Melissa M. Lee

**Journal:** American Political Science Review (2026)

This notebook reproduces the analysis from the paper above.
It was auto-generated by [PoliRep](https://polirep.org).

---

## Setup

Install packages and import standard libraries.

In [ ]:
!pip install -q pandas numpy matplotlib scipy statsmodels pyfixest

import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

## Source Modules

Write the paper's `src/` package to disk so `from src.* import ...` works.

In [ ]:
import sys
from pathlib import Path

# Write src/ package to Colab filesystem
Path("src").mkdir(exist_ok=True)
Path("src/__init__.py").write_text("")
Path("src/config.py").write_text("\"\"\"Configuration: paths and constants for this paper's reproduction.\"\"\"\n\nfrom pathlib import Path\n\n# ---------------------------------------------------------------------------\n# Directory structure\n# ---------------------------------------------------------------------------\n\n# papers/<slug>/\nPAPER_ROOT = Path(\".\")\n\n# Top-level project root\nPROJECT_ROOT = PAPER_ROOT\n\n# Data paths\nDATA_RAW = Path(\"data/raw\")\nDATA_CONVERTED = Path(\"data/converted\")\n\n# Output paths\nOUTPUT_DIR = Path(\"output\")\nTABLE_DIR = OUTPUT_DIR / \"tables\"\nFIGURE_DIR = OUTPUT_DIR / \"figures\"\n\n# Original outputs for comparison\nORIGINAL_TABLE_DIR = Path(\"original/tables\")\nORIGINAL_FIGURE_DIR = Path(\"original/figures\")\n\n\n# ---------------------------------------------------------------------------\n# Analysis constants\n# ---------------------------------------------------------------------------\n\n# Birth cohorts\nBIRTH_COHORTS = [1949, 1950, 1951, 1952]  # Main analysis cohorts\nALL_COHORTS = [1948, 1949, 1950, 1951, 1952, 1953]  # Extended range\n\n# Sample definitions\nSAMPLES = {\n    \"pooled\": \"sample_main1\",  # All origins, 1949-1952\n    \"western\": \"sample_main2\",  # Western origin, 1949-1952\n    \"nonwestern\": \"sample_main3\",  # Non-Western origin, 1949-1952\n}\n\n# Outcome variables (integration measures)\nPRIMARY_OUTCOMES = [\n    \"naturalized\",\n    \"res_integrate_tract\",\n    \"res_integrate_blkgrp\",\n    \"only_engl\",\n    \"engl_ability\",\n    \"spouse_notconatl\",\n    \"spouse_native\",\n]\n\nSECONDARY_OUTCOMES = [\n    \"married\",\n    \"spouse_whitenative\",\n    \"college_some\",\n    \"college_grad\",\n    \"income_total\",\n    \"unemployed\",\n]\n\n# Treatment and instrument\nTREATMENT_VAR = \"veteran\"\nINSTRUMENT_VAR = \"draft_risk\"\n\n# Fixed effect variable\nPANEL_VAR = \"bpl\"  # Birthplace (country of origin)\n\n# Control variables\nCONTROLS = [\"age_immig\"]  # Plus birth year/month dummies\n\n# Simulation parameters (for restricted data context)\nSIMULATION = {\n    \"n_obs_pooled\": 28500,\n    \"n_obs_western\": 9700,\n    \"n_obs_nonwestern\": 18800,\n    \"veteran_rate\": 0.16,\n    \"compliance_rate\": 0.07,\n    \"n_birthplaces\": 120,\n}\n\n# Data validation tolerances\nTOLERANCE = {\n    \"strict\": 1e-6,  # For exact matches\n    \"loose\": 1e-3,  # For coefficient comparisons\n    \"relaxed\": 0.05,  # For summary statistics\n}\n\n\n# ---------------------------------------------------------------------------\n# Helpers\n# ---------------------------------------------------------------------------\n\ndef ensure_output_dirs():\n    \"\"\"Create output directories if they don't exist.\"\"\"\n    TABLE_DIR.mkdir(parents=True, exist_ok=True)\n    FIGURE_DIR.mkdir(parents=True, exist_ok=True)\n")
Path("src/data.py").write_text("\"\"\"Data loading and sample construction.\n\nPortal compatibility requires these three functions:\n  - load_main_data() -> pd.DataFrame\n  - prepare_analysis_sample(df) -> pd.DataFrame\n  - get_sample_stats(sample) -> dict\n\nNOTE: This paper uses restricted Census Bureau microdata available only at\nFederal Statistical Research Data Centers (FSRDC). Since the actual data\ncannot be distributed, this module generates simulated data that mimics\nthe structure and relationships in the original analysis.\n\"\"\"\n\nimport warnings\n\nimport numpy as np\nimport pandas as pd\n\ntry:\n    from .config import BIRTH_COHORTS, DATA_CONVERTED, PANEL_VAR, SIMULATION\nexcept ImportError:\n    from config import BIRTH_COHORTS, DATA_CONVERTED, PANEL_VAR, SIMULATION\n\n\ndef load_main_data() -> pd.DataFrame:\n    \"\"\"Load or generate the main analysis dataset.\n\n    For restricted-access papers, this function attempts to load real data\n    if available (e.g., for users with FSRDC access), otherwise generates\n    simulated data for demonstration.\n\n    Returns:\n        DataFrame with all variables needed for analysis.\n    \"\"\"\n    # Check if actual data exists\n    analysis_file = DATA_CONVERTED / \"analysis.parquet\"\n\n    if analysis_file.exists():\n        warnings.warn(\"Loading actual restricted data - ensure FSRDC approval\")\n        return pd.read_parquet(analysis_file)\n    else:\n        warnings.warn(\n            \"Restricted data not available - generating simulated data. \"\n            \"Results are for demonstration only and do not reflect actual findings.\"\n        )\n        return _generate_simulated_data()\n\n\ndef _generate_simulated_data() -> pd.DataFrame:\n    \"\"\"Generate simulated data that mimics Census 2000 microdata structure.\n\n    This creates a synthetic dataset with realistic correlations between\n    draft risk, military service, and integration outcomes.\n    \"\"\"\n    np.random.seed(42)\n    n = SIMULATION[\"n_obs_pooled\"]\n\n    # Core identifiers\n    df = pd.DataFrame({\"id\": range(1, n + 1)})\n\n    # Birth cohort (stratified sampling)\n    df[\"birth_year\"] = np.random.choice(BIRTH_COHORTS, size=n, p=[0.25] * 4)\n    df[\"birth_month\"] = np.random.randint(1, 13, size=n)\n    df[\"birth_day\"] = np.random.randint(1, 29, size=n)  # Simplified\n\n    # Birthplace (country of origin) - panel variable\n    # Use realistic distribution: more from Mexico, Canada, Western Europe\n    bpl_codes = np.arange(100, 100 + SIMULATION[\"n_birthplaces\"])\n    bpl_weights = np.exp(-np.linspace(0, 3, SIMULATION[\"n_birthplaces\"]))\n    bpl_weights = bpl_weights / bpl_weights.sum()\n    df[PANEL_VAR] = np.random.choice(bpl_codes, size=n, p=bpl_weights)\n\n    # Western vs non-Western origin (roughly 34% Western)\n    df[\"origin\"] = (df[PANEL_VAR] < 160).astype(int)\n\n    # Draft lottery: Random Sequence Number (1-366) and cutoff\n    df[\"rsn\"] = np.random.randint(1, 367, size=n)\n    # Cutoffs varied by year: ~195 in 1969-70, higher later\n    cutoff_map = {1949: 195, 1950: 195, 1951: 215, 1952: 230}\n    df[\"apn\"] = df[\"birth_year\"].map(cutoff_map)\n    df[\"draft_risk\"] = (df[\"rsn\"] <= df[\"apn\"]).astype(int)\n\n    # Veteran status (complier structure with ~7% compliance rate)\n    # P(vet | draft_risk=1) ~ 0.23, P(vet | draft_risk=0) ~ 0.16\n    df[\"veteran\"] = np.where(\n        df[\"draft_risk\"] == 1,\n        np.random.binomial(1, 0.23, size=n),\n        np.random.binomial(1, 0.16, size=n),\n    )\n\n    # Immigration timing\n    df[\"yr2us\"] = np.random.randint(1950, 1970, size=n)  # Before draft\n    df[\"age_immig\"] = df[\"yr2us\"] - df[\"birth_year\"]\n    df[\"yrs_since_immig\"] = 2000 - df[\"yr2us\"]\n\n    # Integration outcomes (correlated with veteran status)\n    # Effect sizes calibrated to plausible IV estimates\n    vet_effect = df[\"veteran\"].values\n\n    # Naturalization (binary) - positive effect\n    base_nat = 0.65\n    df[\"naturalized\"] = np.random.binomial(1, base_nat + 0.10 * vet_effect, size=n)\n\n    # Residential integration (continuous, 0-1) - positive effect\n    base_res = 0.75\n    df[\"res_integrate_tract\"] = np.clip(\n        base_res + 0.08 * vet_effect + np.random.normal(0, 0.15, size=n), 0, 1\n    )\n    df[\"res_integrate_blkgrp\"] = np.clip(\n        0.70 + 0.08 * vet_effect + np.random.normal(0, 0.15, size=n), 0, 1\n    )\n\n    # Language (binary) - positive effect\n    base_engl = 0.45\n    df[\"only_engl\"] = np.random.binomial(1, base_engl + 0.12 * vet_effect, size=n)\n\n    # English ability (ordinal 0-4) - positive effect\n    base_ability = 2.5\n    ability_latent = base_ability + 0.4 * vet_effect + np.random.normal(0, 1, size=n)\n    df[\"engl_ability\"] = np.clip(np.round(ability_latent), 0, 4).astype(int)\n\n    # Marriage outcomes\n    df[\"married\"] = np.random.binomial(1, 0.72, size=n)\n    # Spouse outcomes conditional on marriage\n    df[\"spouse_notconatl\"] = np.where(\n        df[\"married\"] == 1,\n        np.random.binomial(1, 0.55 + 0.15 * vet_effect, size=n),\n        np.nan,\n    )\n    df[\"spouse_native\"] = np.where(\n        df[\"married\"] == 1,\n        np.random.binomial(1, 0.38 + 0.12 * vet_effect, size=n),\n        np.nan,\n    )\n    df[\"spouse_whitenative\"] = np.where(\n        df[\"married\"] == 1, np.random.binomial(1, 0.25, size=n), np.nan\n    )\n\n    # Socioeconomic outcomes\n    df[\"college_some\"] = np.random.binomial(1, 0.42 + 0.08 * vet_effect, size=n)\n    df[\"college_grad\"] = np.random.binomial(1, 0.22 + 0.05 * vet_effect, size=n)\n    df[\"income_total\"] = np.clip(\n        4.5 + 0.8 * vet_effect + np.random.normal(0, 2.5, size=n), 0, None\n    )\n    df[\"unemployed\"] = np.random.binomial(1, 0.06 - 0.02 * vet_effect, size=n)\n\n    # Sample indicators\n    df[\"sample_main1\"] = (df[\"birth_year\"] >= 1949) & (df[\"birth_year\"] <= 1952)\n    df[\"sample_main2\"] = df[\"sample_main1\"] & (df[\"origin\"] == 1)\n    df[\"sample_main3\"] = df[\"sample_main1\"] & (df[\"origin\"] == 0)\n\n    # Person weight (equal weights for simulation)\n    df[\"pwt\"] = 1.0\n\n    return df\n\n\ndef prepare_analysis_sample(df: pd.DataFrame, sample: str = \"pooled\") -> pd.DataFrame:\n    \"\"\"Apply sample restrictions and construct derived variables.\n\n    Args:\n        df: Raw loaded DataFrame from load_main_data().\n        sample: Which sample to use - \"pooled\", \"western\", or \"nonwestern\"\n\n    Returns:\n        Analysis-ready DataFrame with all filters and transformations applied.\n    \"\"\"\n    try:\n        from .config import SAMPLES\n    except ImportError:\n        from config import SAMPLES\n\n    # Select appropriate sample\n    sample_var = SAMPLES[sample]\n    analysis = df[df[sample_var] == True].copy()  # noqa: E712\n\n    # Create birth year and month dummies for FE\n    for year in BIRTH_COHORTS:\n        analysis[f\"byear_{year}\"] = (analysis[\"birth_year\"] == year).astype(int)\n\n    for month in range(1, 13):\n        analysis[f\"bmonth_{month}\"] = (analysis[\"birth_month\"] == month).astype(int)\n\n    return analysis\n\n\ndef get_sample_stats(sample: pd.DataFrame) -> dict:\n    \"\"\"Compute summary statistics for the analysis sample.\n\n    Args:\n        sample: Prepared analysis sample from prepare_analysis_sample().\n\n    Returns:\n        Dictionary with keys like n_obs, n_units, year_range, etc.\n    \"\"\"\n    stats = {\n        \"n_obs\": len(sample),\n        \"n_birthplaces\": sample[PANEL_VAR].nunique(),\n        \"year_range\": f\"{sample['birth_year'].min()}-{sample['birth_year'].max()}\",\n        \"veteran_rate\": sample[\"veteran\"].mean(),\n        \"draft_risk_rate\": sample[\"draft_risk\"].mean(),\n        \"compliance_rate\": (\n            sample[sample[\"draft_risk\"] == 1][\"veteran\"].mean()\n            - sample[sample[\"draft_risk\"] == 0][\"veteran\"].mean()\n        ),\n    }\n    return stats\n")
Path("src/descriptives.py").write_text("\"\"\"Descriptive statistics and summary tables.\n\nReplicates Table A4.1 (sample descriptives) and Table 1 (cross-tabulations).\n\"\"\"\n\nimport pandas as pd\n\nfrom .config import PRIMARY_OUTCOMES, SECONDARY_OUTCOMES, TABLE_DIR, TREATMENT_VAR\nfrom .helpers import compute_summary_stats, save_table\n\n\ndef run(sample: pd.DataFrame = None, sample_name: str = \"pooled\") -> dict:\n    \"\"\"Compute descriptive statistics for the analysis sample.\n\n    Args:\n        sample: Analysis sample DataFrame (if None, will load from data module)\n        sample_name: Sample identifier for output files\n\n    Returns:\n        Dictionary with summary statistics\n    \"\"\"\n    if sample is None:\n        from .data import load_main_data, prepare_analysis_sample\n\n        df = load_main_data()\n        sample = prepare_analysis_sample(df, sample=sample_name)\n\n    print(f\"\\n{'='*60}\")\n    print(f\"Descriptive Statistics: {sample_name.upper()}\")\n    print(f\"{'='*60}\\n\")\n\n    # Basic sample characteristics\n    print(f\"Total observations: {len(sample):,}\")\n    print(f\"Birth years: {sample['birth_year'].min()}-{sample['birth_year'].max()}\")\n    print(f\"Unique birthplaces: {sample['bpl'].nunique()}\")\n    print(f\"Veteran rate: {sample[TREATMENT_VAR].mean():.3f}\")\n    print(f\"Draft risk rate: {sample['draft_risk'].mean():.3f}\")\n\n    compliance = (\n        sample[sample[\"draft_risk\"] == 1][TREATMENT_VAR].mean()\n        - sample[sample[\"draft_risk\"] == 0][TREATMENT_VAR].mean()\n    )\n    print(f\"Compliance rate: {compliance:.3f}\\n\")\n\n    # Variables to summarize\n    key_vars = [\n        TREATMENT_VAR,\n        \"draft_risk\",\n        \"age_immig\",\n        \"yrs_since_immig\",\n    ] + PRIMARY_OUTCOMES + SECONDARY_OUTCOMES\n\n    # Filter to variables that exist\n    key_vars = [v for v in key_vars if v in sample.columns]\n\n    # Compute summary statistics\n    print(\"Computing summary statistics...\")\n    stats = compute_summary_stats(sample, key_vars, weights=\"pwt\")\n\n    print(\"\\n\" + \"=\"*80)\n    print(stats.to_string(index=False))\n    print(\"=\"*80)\n\n    # Save to CSV and LaTeX\n    csv_file = f\"descriptives_{sample_name}.csv\"\n    stats.to_csv(TABLE_DIR / csv_file, index=False)\n    print(f\"\\nSaved: {TABLE_DIR / csv_file}\")\n\n    # Create LaTeX table\n    latex = stats.to_latex(\n        index=False,\n        float_format=\"%.3f\",\n        caption=f\"Descriptive Statistics: {sample_name.title()} Sample\",\n        label=f\"tab:desc_{sample_name}\",\n    )\n    tex_file = f\"descriptives_{sample_name}.tex\"\n    save_table(latex, tex_file, TABLE_DIR)\n\n    # Cross-tabulation: Veteran status by draft risk\n    print(\"\\n\" + \"=\"*60)\n    print(\"CROSS-TABULATION: Veteran Status by Draft Risk\")\n    print(\"=\"*60)\n\n    crosstab = pd.crosstab(\n        sample[\"draft_risk\"],\n        sample[TREATMENT_VAR],\n        normalize=\"index\",\n        margins=True,\n    )\n    print(crosstab)\n\n    crosstab_counts = pd.crosstab(\n        sample[\"draft_risk\"],\n        sample[TREATMENT_VAR],\n        margins=True,\n    )\n    print(\"\\nCounts:\")\n    print(crosstab_counts)\n\n    # Save crosstab\n    crosstab_file = f\"crosstab_veteran_{sample_name}.csv\"\n    crosstab_counts.to_csv(TABLE_DIR / crosstab_file)\n    print(f\"\\nSaved: {TABLE_DIR / crosstab_file}\")\n\n    return {\n        \"stats\": stats,\n        \"crosstab\": crosstab,\n        \"n_obs\": len(sample),\n        \"compliance_rate\": compliance,\n    }\n\n\nif __name__ == \"__main__\":\n    # Run for all three samples\n    for sample_name in [\"pooled\", \"western\", \"nonwestern\"]:\n        run(sample_name=sample_name)\n")
Path("src/first_stage.py").write_text("\"\"\"First-stage analysis: Draft risk → Veteran status.\n\nReplicates Table A6.2 (first stage by sample) and Table 1 last column (compliers).\n\"\"\"\n\nimport pandas as pd\n\nfrom .config import BIRTH_COHORTS, CONTROLS, INSTRUMENT_VAR, PANEL_VAR, TABLE_DIR, TREATMENT_VAR\nfrom .helpers import first_stage_diagnostics\n\n\ndef run(sample: pd.DataFrame = None, sample_name: str = \"pooled\") -> dict:\n    \"\"\"Run first-stage analysis.\n\n    Args:\n        sample: Analysis sample DataFrame (if None, will load from data module)\n        sample_name: Sample identifier for output files\n\n    Returns:\n        Dictionary with results\n    \"\"\"\n    if sample is None:\n        from .data import load_main_data, prepare_analysis_sample\n\n        df = load_main_data()\n        sample = prepare_analysis_sample(df, sample=sample_name)\n\n    print(f\"\\n{'='*60}\")\n    print(f\"First Stage Analysis: {sample_name.upper()}\")\n    print(f\"{'='*60}\\n\")\n\n    # Build control list (age_immig + birth year/month dummies)\n    controls = CONTROLS.copy()\n    for year in BIRTH_COHORTS:\n        controls.append(f\"byear_{year}\")\n    for month in range(1, 13):\n        controls.append(f\"bmonth_{month}\")\n\n    # Remove one category from each set of dummies to avoid collinearity\n    controls.remove(\"byear_1949\")\n    controls.remove(\"bmonth_1\")\n\n    # Run first stage\n    print(f\"Dependent variable: {TREATMENT_VAR}\")\n    print(f\"Instrument: {INSTRUMENT_VAR}\")\n    print(\"Controls: age_immig + birth year FE + birth month FE\")\n    print(f\"Entity effects: {PANEL_VAR} (birthplace)\")\n    print(f\"Sample size: {len(sample):,}\\n\")\n\n    result = first_stage_diagnostics(\n        data=sample,\n        treatment=TREATMENT_VAR,\n        instrument=INSTRUMENT_VAR,\n        controls=controls,\n        entity_effects=PANEL_VAR,\n    )\n\n    # Extract results\n    model = result[\"model\"]\n    f_stat = result[\"f_statistic\"]\n    coef = result[\"coef\"]\n\n    print(f\"First-stage coefficient on {INSTRUMENT_VAR}: {coef:.4f}\")\n    if hasattr(model, \"std_errors\"):\n        se = model.std_errors.get(INSTRUMENT_VAR, None)\n        if se:\n            print(f\"Standard error: {se:.4f}\")\n            print(f\"t-statistic: {coef/se:.2f}\")\n\n    if f_stat:\n        print(f\"F-statistic: {f_stat:.2f}\")\n\n    # Compute compliance rate (reduced form)\n    compliance = (\n        sample[sample[INSTRUMENT_VAR] == 1][TREATMENT_VAR].mean()\n        - sample[sample[INSTRUMENT_VAR] == 0][TREATMENT_VAR].mean()\n    )\n    print(f\"Compliance rate (LATE denominator): {compliance:.4f}\")\n\n    # Save results to CSV\n    results_df = pd.DataFrame(\n        {\n            \"Sample\": [sample_name],\n            \"N\": [len(sample)],\n            \"Draft_Risk_Coef\": [coef],\n            \"F_Statistic\": [f_stat if f_stat else \"N/A\"],\n            \"Compliance_Rate\": [compliance],\n        }\n    )\n\n    output_file = f\"first_stage_{sample_name}.csv\"\n    results_df.to_csv(TABLE_DIR / output_file, index=False)\n    print(f\"\\nSaved: {TABLE_DIR / output_file}\")\n\n    return {\n        \"model\": model,\n        \"coefficient\": coef,\n        \"f_statistic\": f_stat,\n        \"compliance_rate\": compliance,\n        \"n_obs\": len(sample),\n    }\n\n\nif __name__ == \"__main__\":\n    # Run for all three samples\n    for sample_name in [\"pooled\", \"western\", \"nonwestern\"]:\n        run(sample_name=sample_name)\n")
Path("src/helpers.py").write_text("\"\"\"Helper functions for IV estimation and table formatting.\"\"\"\n\nimport numpy as np\nimport pandas as pd\nfrom linearmodels.iv import IV2SLS\nfrom scipy import stats\nfrom statsmodels.iolib.summary2 import summary_col\n\n\ndef run_2sls(\n    data: pd.DataFrame,\n    outcome: str,\n    treatment: str,\n    instrument: str,\n    controls: list,\n    entity_effects: str = None,\n) -> IV2SLS:\n    \"\"\"Run 2SLS regression with entity fixed effects.\n\n    Args:\n        data: Analysis DataFrame\n        outcome: Dependent variable name\n        treatment: Endogenous treatment variable\n        instrument: Instrumental variable\n        controls: List of control variable names\n        entity_effects: Variable name for entity (country) fixed effects\n\n    Returns:\n        Fitted IV2SLS model\n    \"\"\"\n\n    # Drop missing values\n    vars_needed = [outcome, treatment, instrument] + controls\n    if entity_effects:\n        vars_needed.append(entity_effects)\n\n    clean_data = data[vars_needed].dropna().copy()\n\n    # If entity effects requested, create dummies manually\n    if entity_effects:\n        # Get dummies for entity effects (drop one category)\n        entity_dummies = pd.get_dummies(clean_data[entity_effects], prefix='fe', drop_first=True)\n        # Add to controls\n        exog_vars = controls + list(entity_dummies.columns)\n        clean_data = pd.concat([clean_data, entity_dummies], axis=1)\n    else:\n        exog_vars = controls\n\n    # Set up dependent variable\n    y = clean_data[outcome]\n\n    # Set up exogenous variables (controls + entity FE dummies)\n    from statsmodels.api import add_constant\n\n    if exog_vars:\n        X = clean_data[exog_vars]\n        X = add_constant(X)\n    else:\n        X = add_constant(pd.DataFrame(index=clean_data.index))\n\n    # Set up endogenous variable\n    endog = clean_data[[treatment]]\n\n    # Set up instrument\n    instruments = clean_data[[instrument]]\n\n    # Run IV2SLS\n    model = IV2SLS(\n        dependent=y,\n        exog=X,\n        endog=endog,\n        instruments=instruments,\n    )\n\n    result = model.fit(cov_type=\"robust\")\n    return result\n\n\ndef run_ols(\n    data: pd.DataFrame,\n    outcome: str,\n    treatment: str,\n    controls: list,\n    entity_effects: str = None,\n):\n    \"\"\"Run OLS regression (reduced form or naive).\n\n    Args:\n        data: Analysis DataFrame\n        outcome: Dependent variable name\n        treatment: Treatment variable\n        controls: List of control variable names\n        entity_effects: Variable name for entity fixed effects\n\n    Returns:\n        Fitted model\n    \"\"\"\n    from linearmodels.panel import PanelOLS\n\n    # Drop missing values\n    vars_needed = [outcome, treatment] + controls\n    if entity_effects:\n        vars_needed.append(entity_effects)\n\n    clean_data = data[vars_needed].dropna()\n\n    if entity_effects:\n        # Use panel OLS with entity effects\n        # Need to set multi-index\n        clean_data = clean_data.set_index([entity_effects, clean_data.index])\n\n        # Formula\n        formula_parts = [treatment] + controls\n        formula = outcome + \" ~ \" + \" + \".join(formula_parts) + \" + EntityEffects\"\n\n        model = PanelOLS.from_formula(formula, data=clean_data)\n        result = model.fit(cov_type=\"robust\")\n    else:\n        # Simple OLS\n        from statsmodels.api import OLS, add_constant\n\n        X = clean_data[[treatment] + controls]\n        X = add_constant(X)\n        y = clean_data[outcome]\n\n        model = OLS(y, X)\n        result = model.fit(cov_type=\"HC1\")\n\n    return result\n\n\ndef first_stage_diagnostics(\n    data: pd.DataFrame, treatment: str, instrument: str, controls: list, entity_effects: str = None\n) -> dict:\n    \"\"\"Compute first-stage regression and diagnostics.\n\n    Args:\n        data: Analysis DataFrame\n        treatment: Endogenous variable\n        instrument: Instrumental variable\n        controls: List of controls\n        entity_effects: Entity FE variable\n\n    Returns:\n        Dictionary with first-stage results and F-statistic\n    \"\"\"\n    # Run first stage regression\n    result = run_ols(data, treatment, instrument, controls, entity_effects)\n\n    # Extract F-statistic for instrument\n    # For IV with one instrument, this is the first-stage F-stat\n    fs_fstat = result.fvalue if hasattr(result, \"fvalue\") else None\n\n    return {\"model\": result, \"f_statistic\": fs_fstat, \"coef\": result.params.get(instrument, np.nan)}\n\n\ndef format_regression_table(models: list, model_names: list, outcome_name: str) -> str:\n    \"\"\"Format regression results as LaTeX table.\n\n    Args:\n        models: List of fitted models\n        model_names: List of model labels\n        outcome_name: Name of the outcome variable\n\n    Returns:\n        LaTeX table string\n    \"\"\"\n    # Use statsmodels summary_col\n    table = summary_col(\n        models,\n        model_names=model_names,\n        stars=True,\n        float_format=\"%.4f\",\n        info_dict={\n            \"N\": lambda x: f\"{int(x.nobs):,}\",\n            \"R-squared\": lambda x: f\"{x.rsquared:.3f}\" if hasattr(x, \"rsquared\") else \"\",\n        },\n    )\n\n    latex = table.as_latex()\n    return latex\n\n\ndef compute_summary_stats(data: pd.DataFrame, variables: list, weights: str = None) -> pd.DataFrame:\n    \"\"\"Compute summary statistics for specified variables.\n\n    Args:\n        data: Analysis DataFrame\n        variables: List of variable names\n        weights: Optional weight variable\n\n    Returns:\n        DataFrame with mean, SD, min, max, N\n    \"\"\"\n    stats_list = []\n\n    for var in variables:\n        var_data = data[var].dropna()\n\n        if weights and weights in data.columns:\n            wt = data.loc[var_data.index, weights]\n            mean = np.average(var_data, weights=wt)\n            # Weighted variance\n            variance = np.average((var_data - mean) ** 2, weights=wt)\n            sd = np.sqrt(variance)\n        else:\n            mean = var_data.mean()\n            sd = var_data.std()\n\n        stats_list.append(\n            {\n                \"Variable\": var,\n                \"Mean\": mean,\n                \"SD\": sd,\n                \"Min\": var_data.min(),\n                \"Max\": var_data.max(),\n                \"N\": len(var_data),\n            }\n        )\n\n    return pd.DataFrame(stats_list)\n\n\ndef proportion_test(observed: float, expected: float, n: int) -> dict:\n    \"\"\"One-sample proportion test.\n\n    Args:\n        observed: Observed proportion\n        expected: Expected proportion under null\n        n: Sample size\n\n    Returns:\n        Dictionary with z-statistic and p-value\n    \"\"\"\n    # One-sample proportion test\n    z_stat = (observed - expected) / np.sqrt(expected * (1 - expected) / n)\n    p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))\n\n    return {\"z_statistic\": z_stat, \"p_value\": p_value, \"observed\": observed, \"expected\": expected}\n\n\ndef save_table(content: str, filename: str, output_dir):\n    \"\"\"Save table to file.\n\n    Args:\n        content: Table content (LaTeX or CSV)\n        filename: Output filename\n        output_dir: Output directory path\n    \"\"\"\n    filepath = output_dir / filename\n    with open(filepath, \"w\") as f:\n        f.write(content)\n    print(f\"Saved: {filepath}\")\n")
Path("src/main_iv.py").write_text("\"\"\"Main IV analysis: Causal effect of military service on integration.\n\nReplicates Tables 2, 3, 4 (main IV results for pooled, Western, Non-Western samples).\n\"\"\"\n\nimport pandas as pd\n\nfrom .config import (\n    BIRTH_COHORTS,\n    CONTROLS,\n    INSTRUMENT_VAR,\n    PANEL_VAR,\n    PRIMARY_OUTCOMES,\n    TABLE_DIR,\n    TREATMENT_VAR,\n)\nfrom .helpers import run_2sls, run_ols\n\n\ndef run(sample: pd.DataFrame = None, sample_name: str = \"pooled\") -> dict:\n    \"\"\"Run main IV analysis for all integration outcomes.\n\n    Args:\n        sample: Analysis sample DataFrame (if None, will load from data module)\n        sample_name: Sample identifier for output files\n\n    Returns:\n        Dictionary with results for each outcome\n    \"\"\"\n    if sample is None:\n        from .data import load_main_data, prepare_analysis_sample\n\n        df = load_main_data()\n        sample = prepare_analysis_sample(df, sample=sample_name)\n\n    print(f\"\\n{'='*60}\")\n    print(f\"Main IV Analysis: {sample_name.upper()}\")\n    print(f\"{'='*60}\\n\")\n\n    # Build control list\n    controls = CONTROLS.copy()\n    for year in BIRTH_COHORTS:\n        controls.append(f\"byear_{year}\")\n    for month in range(1, 13):\n        controls.append(f\"bmonth_{month}\")\n\n    # Remove reference categories\n    controls.remove(\"byear_1949\")\n    controls.remove(\"bmonth_1\")\n\n    # Store results\n    results = {}\n\n    # For each outcome, run OLS, ITT (reduced form), and 2SLS\n    for outcome in PRIMARY_OUTCOMES:\n        print(f\"\\nOutcome: {outcome}\")\n        print(\"-\" * 40)\n\n        # Check if outcome exists and has variation\n        if outcome not in sample.columns:\n            print(f\"  WARNING: {outcome} not in data, skipping\")\n            continue\n\n        outcome_data = sample[outcome].dropna()\n        if len(outcome_data) < 100:\n            print(f\"  WARNING: Too few observations ({len(outcome_data)}), skipping\")\n            continue\n\n        # (1) Naive OLS: outcome ~ veteran + controls + bpl FE\n        print(\"  Running OLS...\")\n        try:\n            ols_result = run_ols(sample, outcome, TREATMENT_VAR, controls, PANEL_VAR)\n            ols_coef = ols_result.params.get(TREATMENT_VAR, None)\n            ols_se = ols_result.std_errors.get(TREATMENT_VAR, None) if hasattr(\n                ols_result, \"std_errors\"\n            ) else None\n            print(f\"    OLS coef: {ols_coef:.4f} ({ols_se:.4f})\")\n        except Exception as e:\n            print(f\"    OLS failed: {e}\")\n            ols_result = None\n            ols_coef = None\n            ols_se = None\n\n        # (2) ITT (reduced form): outcome ~ draft_risk + controls + bpl FE\n        print(\"  Running ITT (reduced form)...\")\n        try:\n            itt_result = run_ols(sample, outcome, INSTRUMENT_VAR, controls, PANEL_VAR)\n            itt_coef = itt_result.params.get(INSTRUMENT_VAR, None)\n            itt_se = itt_result.std_errors.get(INSTRUMENT_VAR, None) if hasattr(\n                itt_result, \"std_errors\"\n            ) else None\n            print(f\"    ITT coef: {itt_coef:.4f} ({itt_se:.4f})\")\n        except Exception as e:\n            print(f\"    ITT failed: {e}\")\n            itt_result = None\n            itt_coef = None\n            itt_se = None\n\n        # (3) 2SLS: outcome ~ veteran (instrumented) + controls + bpl FE\n        print(\"  Running 2SLS...\")\n        try:\n            iv_result = run_2sls(\n                sample, outcome, TREATMENT_VAR, INSTRUMENT_VAR, controls, PANEL_VAR\n            )\n            iv_coef = iv_result.params.get(TREATMENT_VAR, None)\n            iv_se = iv_result.std_errors.get(TREATMENT_VAR, None)\n            print(f\"    2SLS coef: {iv_coef:.4f} ({iv_se:.4f})\")\n        except Exception as e:\n            print(f\"    2SLS failed: {e}\")\n            iv_result = None\n            iv_coef = None\n            iv_se = None\n\n        # Store results\n        results[outcome] = {\n            \"ols_model\": ols_result,\n            \"ols_coef\": ols_coef,\n            \"ols_se\": ols_se,\n            \"itt_model\": itt_result,\n            \"itt_coef\": itt_coef,\n            \"itt_se\": itt_se,\n            \"iv_model\": iv_result,\n            \"iv_coef\": iv_coef,\n            \"iv_se\": iv_se,\n        }\n\n    # Create summary table\n    print(\"\\n\" + \"=\"*60)\n    print(\"SUMMARY TABLE\")\n    print(\"=\"*60)\n\n    summary_rows = []\n    for outcome, res in results.items():\n        row = {\n            \"Outcome\": outcome,\n            \"OLS_Coef\": res[\"ols_coef\"],\n            \"OLS_SE\": res[\"ols_se\"],\n            \"ITT_Coef\": res[\"itt_coef\"],\n            \"ITT_SE\": res[\"itt_se\"],\n            \"IV_Coef\": res[\"iv_coef\"],\n            \"IV_SE\": res[\"iv_se\"],\n        }\n        summary_rows.append(row)\n\n    summary_df = pd.DataFrame(summary_rows)\n    print(summary_df.to_string(index=False))\n\n    # Save to CSV\n    output_file = f\"main_iv_{sample_name}.csv\"\n    summary_df.to_csv(TABLE_DIR / output_file, index=False)\n    print(f\"\\nSaved: {TABLE_DIR / output_file}\")\n\n    return results\n\n\nif __name__ == \"__main__\":\n    # Run for all three samples\n    for sample_name in [\"pooled\", \"western\", \"nonwestern\"]:\n        run(sample_name=sample_name)\n")
Path("src/robustness.py").write_text("\"\"\"Robustness checks and supplementary analyses.\n\nReplicates:\n- Table A8.4 (excess mortality test)\n- Table A8.5 (attrition by nationality)\n- Table A10.6 (exclusion restriction tests)\n\"\"\"\n\nimport pandas as pd\n\nfrom .config import BIRTH_COHORTS, CONTROLS, INSTRUMENT_VAR, PANEL_VAR, TABLE_DIR, TREATMENT_VAR\nfrom .helpers import proportion_test\n\n\ndef run(sample: pd.DataFrame = None, sample_name: str = \"pooled\") -> dict:\n    \"\"\"Run robustness checks.\n\n    Args:\n        sample: Analysis sample DataFrame (if None, will load from data module)\n        sample_name: Sample identifier for output files\n\n    Returns:\n        Dictionary with robustness test results\n    \"\"\"\n    if sample is None:\n        from .data import load_main_data, prepare_analysis_sample\n\n        df = load_main_data()\n        sample = prepare_analysis_sample(df, sample=sample_name)\n\n    print(f\"\\n{'='*60}\")\n    print(f\"Robustness Checks: {sample_name.upper()}\")\n    print(f\"{'='*60}\\n\")\n\n    results = {}\n\n    # -------------------------------------------------------------------------\n    # 1. Excess mortality test (Table A8.4)\n    # -------------------------------------------------------------------------\n    print(\"\\n\" + \"-\"*60)\n    print(\"1. EXCESS MORTALITY TEST\")\n    print(\"-\"*60)\n    print(\"Testing whether veterans have higher mortality than expected.\")\n    print(\"(In Census 2000, measuring survival to 2000 for Vietnam-era cohorts)\\n\")\n\n    # For demonstration: assume we can calculate expected survival\n    # In reality, this requires external mortality tables\n    veteran_rate = sample[TREATMENT_VAR].mean()\n    expected_rate = 0.16  # Hypothetical baseline from non-combat data\n\n    n_obs = len(sample)\n    mortality_test = proportion_test(veteran_rate, expected_rate, n_obs)\n\n    print(f\"Observed veteran rate: {mortality_test['observed']:.4f}\")\n    print(f\"Expected rate (if no excess mortality): {mortality_test['expected']:.4f}\")\n    print(f\"Z-statistic: {mortality_test['z_statistic']:.3f}\")\n    print(f\"P-value: {mortality_test['p_value']:.4f}\")\n\n    if mortality_test['p_value'] > 0.05:\n        print(\"=> No significant excess mortality detected\")\n    else:\n        print(\"=> Significant difference detected (possible attrition bias)\")\n\n    results[\"mortality_test\"] = mortality_test\n\n    # -------------------------------------------------------------------------\n    # 2. Attrition by nationality (Table A8.5)\n    # -------------------------------------------------------------------------\n    print(\"\\n\" + \"-\"*60)\n    print(\"2. ATTRITION BY NATIONALITY\")\n    print(\"-\"*60)\n    print(\"Testing for differential attrition by birthplace.\\n\")\n\n    # For each major birthplace, test if veteran rate differs from pooled\n    top_birthplaces = sample[\"bpl\"].value_counts().head(10).index\n\n    attrition_tests = []\n    for bpl_code in top_birthplaces:\n        bpl_sample = sample[sample[\"bpl\"] == bpl_code]\n        n_bpl = len(bpl_sample)\n\n        if n_bpl < 50:\n            continue\n\n        bpl_vet_rate = bpl_sample[TREATMENT_VAR].mean()\n        test = proportion_test(bpl_vet_rate, veteran_rate, n_bpl)\n\n        attrition_tests.append(\n            {\n                \"Birthplace\": int(bpl_code),\n                \"N\": n_bpl,\n                \"Vet_Rate\": bpl_vet_rate,\n                \"Z_Stat\": test[\"z_statistic\"],\n                \"P_Value\": test[\"p_value\"],\n            }\n        )\n\n    attrition_df = pd.DataFrame(attrition_tests)\n    print(attrition_df.to_string(index=False))\n\n    # Save\n    attrition_file = f\"attrition_by_nationality_{sample_name}.csv\"\n    attrition_df.to_csv(TABLE_DIR / attrition_file, index=False)\n    print(f\"\\nSaved: {TABLE_DIR / attrition_file}\")\n\n    results[\"attrition_tests\"] = attrition_df\n\n    # -------------------------------------------------------------------------\n    # 3. Exclusion restriction: Subgroup analysis (Table A10.6)\n    # -------------------------------------------------------------------------\n    print(\"\\n\" + \"-\"*60)\n    print(\"3. EXCLUSION RESTRICTION TEST\")\n    print(\"-\"*60)\n    print(\"Testing whether draft risk affects outcomes only through service.\\n\")\n    print(\"Strategy: Estimate effect separately for veterans and non-veterans.\")\n    print(\"If exclusion holds, draft_risk should have no effect among non-veterans.\\n\")\n\n    # Build controls\n    controls = CONTROLS.copy()\n    for year in BIRTH_COHORTS:\n        controls.append(f\"byear_{year}\")\n    for month in range(1, 13):\n        controls.append(f\"bmonth_{month}\")\n    controls.remove(\"byear_1949\")\n    controls.remove(\"bmonth_1\")\n\n    # Test on one outcome: naturalization\n    test_outcome = \"naturalized\"\n\n    if test_outcome in sample.columns:\n        # Split by veteran status\n        veterans = sample[sample[TREATMENT_VAR] == 1]\n        non_veterans = sample[sample[TREATMENT_VAR] == 0]\n\n        print(f\"Testing outcome: {test_outcome}\")\n        print(f\"Veterans: N={len(veterans):,}\")\n        print(f\"Non-veterans: N={len(non_veterans):,}\\n\")\n\n        exclusion_results = {}\n\n        # For each subgroup, regress outcome on draft_risk (reduced form)\n        from .helpers import run_ols\n\n        for group_name, group_data in [(\"Veterans\", veterans), (\"Non-Veterans\", non_veterans)]:\n            if len(group_data) < 100:\n                print(f\"{group_name}: Too few observations, skipping\")\n                continue\n\n            try:\n                model = run_ols(group_data, test_outcome, INSTRUMENT_VAR, controls, PANEL_VAR)\n                coef = model.params.get(INSTRUMENT_VAR, None)\n                se = model.std_errors.get(INSTRUMENT_VAR, None) if hasattr(\n                    model, \"std_errors\"\n                ) else None\n\n                print(f\"{group_name}:\")\n                print(f\"  Draft risk coef: {coef:.4f} (SE: {se:.4f})\")\n                print(f\"  T-stat: {coef/se:.2f}\" if se else \"\")\n\n                exclusion_results[group_name] = {\n                    \"coef\": coef,\n                    \"se\": se,\n                    \"n_obs\": len(group_data),\n                }\n            except Exception as e:\n                print(f\"{group_name}: Failed - {e}\")\n\n        # Interpretation\n        if \"Non-Veterans\" in exclusion_results:\n            nv_coef = exclusion_results[\"Non-Veterans\"][\"coef\"]\n            nv_se = exclusion_results[\"Non-Veterans\"][\"se\"]\n            if abs(nv_coef / nv_se) < 1.96:\n                print(\"\\n=> Exclusion restriction holds: No effect among non-veterans\")\n            else:\n                print(\n                    \"\\n=> WARNING: Draft risk affects outcome among non-veterans \"\n                    \"(exclusion restriction violated)\"\n                )\n\n        results[\"exclusion_test\"] = exclusion_results\n\n    return results\n\n\nif __name__ == \"__main__\":\n    # Run for pooled sample\n    run(sample_name=\"pooled\")\n")

# Ensure src is importable
sys.path.insert(0, ".")

# Create output directories
Path("output/tables").mkdir(parents=True, exist_ok=True)
Path("output/figures").mkdir(parents=True, exist_ok=True)

## Data Preparation


### Load Main Data

Load or generate the main analysis dataset. For this restricted-access
paper, attempts to load actual Census 2000 SEDF data if available
(for users with FSRDC access), otherwise generates simulated data.

In [ ]:
def load_main_data() -> pd.DataFrame:
    """Load or generate the main analysis dataset.

    For restricted-access papers, this function attempts to load real data
    if available (e.g., for users with FSRDC access), otherwise generates
    simulated data for demonstration.

    Returns:
        DataFrame with all variables needed for analysis.
    """
    # Check if actual data exists
    analysis_file = DATA_CONVERTED / "analysis.parquet"

    if analysis_file.exists():
        warnings.warn("Loading actual restricted data - ensure FSRDC approval")
        return pd.read_parquet(analysis_file)
    else:
        warnings.warn(
            "Restricted data not available - generating simulated data. "
            "Results are for demonstration only and do not reflect actual findings."
        )
        return _generate_simulated_data()

In [ ]:
df = load_main_data()
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns[:20])}...")
print(f"Birth year range: {int(df['birth_year'].min())}-{int(df['birth_year'].max())}")
print(f"Veteran rate: {df['veteran'].mean():.1%}")
print(f"Draft risk rate: {df['draft_risk'].mean():.1%}")
display(df.head())


### Create Draft Risk Instrument

Merge Vietnam draft lottery data (RSN and APN by birth date) and
create binary draft_risk indicator (1 if RSN ≤ APN). This is the
instrumental variable for veteran status.

In [ ]:
def load_main_data() -> pd.DataFrame:
    """Load or generate the main analysis dataset.

    For restricted-access papers, this function attempts to load real data
    if available (e.g., for users with FSRDC access), otherwise generates
    simulated data for demonstration.

    Returns:
        DataFrame with all variables needed for analysis.
    """
    # Check if actual data exists
    analysis_file = DATA_CONVERTED / "analysis.parquet"

    if analysis_file.exists():
        warnings.warn("Loading actual restricted data - ensure FSRDC approval")
        return pd.read_parquet(analysis_file)
    else:
        warnings.warn(
            "Restricted data not available - generating simulated data. "
            "Results are for demonstration only and do not reflect actual findings."
        )
        return _generate_simulated_data()

In [ ]:
df = load_main_data()
print(f"RSN range: {int(df['rsn'].min())}-{int(df['rsn'].max())}")
print(f"APN by cohort:")
print(df.groupby('birth_year')['apn'].first().to_dict())
print(f"Draft risk rate: {df['draft_risk'].mean():.1%}")
print(f"Compliance rate: {(df[df['draft_risk']==1]['veteran'].mean() - df[df['draft_risk']==0]['veteran'].mean()):.1%}")
display(df[['birth_year', 'birth_month', 'birth_day', 'rsn', 'apn', 'draft_risk']].head(10))


### Create Integration Outcomes

Construct integration outcome variables: naturalization (binary),
residential integration at tract and block group levels (continuous),
English language outcomes (binary and ordinal), and spouse
characteristics (conditional on marriage).

In [ ]:
def load_main_data() -> pd.DataFrame:
    """Load or generate the main analysis dataset.

    For restricted-access papers, this function attempts to load real data
    if available (e.g., for users with FSRDC access), otherwise generates
    simulated data for demonstration.

    Returns:
        DataFrame with all variables needed for analysis.
    """
    # Check if actual data exists
    analysis_file = DATA_CONVERTED / "analysis.parquet"

    if analysis_file.exists():
        warnings.warn("Loading actual restricted data - ensure FSRDC approval")
        return pd.read_parquet(analysis_file)
    else:
        warnings.warn(
            "Restricted data not available - generating simulated data. "
            "Results are for demonstration only and do not reflect actual findings."
        )
        return _generate_simulated_data()

In [ ]:
df = load_main_data()
print("Integration outcome means:")
print(f"  Naturalized: {df['naturalized'].mean():.1%}")
print(f"  Res integrate (tract): {df['res_integrate_tract'].mean():.3f}")
print(f"  Res integrate (blkgrp): {df['res_integrate_blkgrp'].mean():.3f}")
print(f"  Only English: {df['only_engl'].mean():.1%}")
print(f"  English ability (mean): {df['engl_ability'].mean():.2f}")
print(f"  Non-conatl spouse (married): {df['spouse_notconatl'].mean():.1%}")
print(f"  Native spouse (married): {df['spouse_native'].mean():.1%}")
display(df[['naturalized', 'res_integrate_tract', 'only_engl', 'spouse_native']].describe())


### Create Socioeconomic Outcomes

Construct secondary outcomes: marriage (binary), education
(some college, college grad), total income (in $10,000s),
and unemployment (binary).

In [ ]:
def load_main_data() -> pd.DataFrame:
    """Load or generate the main analysis dataset.

    For restricted-access papers, this function attempts to load real data
    if available (e.g., for users with FSRDC access), otherwise generates
    simulated data for demonstration.

    Returns:
        DataFrame with all variables needed for analysis.
    """
    # Check if actual data exists
    analysis_file = DATA_CONVERTED / "analysis.parquet"

    if analysis_file.exists():
        warnings.warn("Loading actual restricted data - ensure FSRDC approval")
        return pd.read_parquet(analysis_file)
    else:
        warnings.warn(
            "Restricted data not available - generating simulated data. "
            "Results are for demonstration only and do not reflect actual findings."
        )
        return _generate_simulated_data()

In [ ]:
df = load_main_data()
print("Socioeconomic outcome means:")
print(f"  Married: {df['married'].mean():.1%}")
print(f"  Some college: {df['college_some'].mean():.1%}")
print(f"  College grad: {df['college_grad'].mean():.1%}")
print(f"  Income total (mean): ${df['income_total'].mean():.1f}k")
print(f"  Unemployed: {df['unemployed'].mean():.1%}")
display(df[['married', 'college_some', 'college_grad', 'income_total']].describe())


### Apply Sample Restrictions

Filter to analysis sample: men born abroad 1949-1952, immigrated
before draft year. Create sample indicators for pooled, Western,
and non-Western subsamples. Create birth year and month dummies
for fixed effects.

In [ ]:
def prepare_analysis_sample(df: pd.DataFrame, sample: str = "pooled") -> pd.DataFrame:
    """Apply sample restrictions and construct derived variables.

    Args:
        df: Raw loaded DataFrame from load_main_data().
        sample: Which sample to use - "pooled", "western", or "nonwestern"

    Returns:
        Analysis-ready DataFrame with all filters and transformations applied.
    """
    try:
        from .config import SAMPLES
    except ImportError:
        from config import SAMPLES

    # Select appropriate sample
    sample_var = SAMPLES[sample]
    analysis = df[df[sample_var] == True].copy()  # noqa: E712

    # Create birth year and month dummies for FE
    for year in BIRTH_COHORTS:
        analysis[f"byear_{year}"] = (analysis["birth_year"] == year).astype(int)

    for month in range(1, 13):
        analysis[f"bmonth_{month}"] = (analysis["birth_month"] == month).astype(int)

    return analysis

def get_sample_stats(sample: pd.DataFrame) -> dict:
    """Compute summary statistics for the analysis sample.

    Args:
        sample: Prepared analysis sample from prepare_analysis_sample().

    Returns:
        Dictionary with keys like n_obs, n_units, year_range, etc.
    """
    stats = {
        "n_obs": len(sample),
        "n_birthplaces": sample[PANEL_VAR].nunique(),
        "year_range": f"{sample['birth_year'].min()}-{sample['birth_year'].max()}",
        "veteran_rate": sample["veteran"].mean(),
        "draft_risk_rate": sample["draft_risk"].mean(),
        "compliance_rate": (
            sample[sample["draft_risk"] == 1]["veteran"].mean()
            - sample[sample["draft_risk"] == 0]["veteran"].mean()
        ),
    }
    return stats

In [ ]:
sample = prepare_analysis_sample(load_main_data(), sample="pooled")
print(f"Pooled sample shape: {sample.shape}")
stats = get_sample_stats(sample)
print(f"Observations: {stats['n_obs']}")
print(f"Birthplaces: {stats['n_birthplaces']}")
print(f"Year range: {stats['year_range']}")
print(f"Veteran rate: {stats['veteran_rate']:.1%}")
print(f"Draft risk rate: {stats['draft_risk_rate']:.1%}")
print(f"Compliance rate: {stats['compliance_rate']:.1%}")
display(sample.head())


### Subsample: Western Origin

Create Western-origin subsample (Europe, Canada, Australia, NZ, Israel).
This subsample has ~9,700 observations and is used for heterogeneity
analysis in Table 3.

In [ ]:
def prepare_analysis_sample(df: pd.DataFrame, sample: str = "pooled") -> pd.DataFrame:
    """Apply sample restrictions and construct derived variables.

    Args:
        df: Raw loaded DataFrame from load_main_data().
        sample: Which sample to use - "pooled", "western", or "nonwestern"

    Returns:
        Analysis-ready DataFrame with all filters and transformations applied.
    """
    try:
        from .config import SAMPLES
    except ImportError:
        from config import SAMPLES

    # Select appropriate sample
    sample_var = SAMPLES[sample]
    analysis = df[df[sample_var] == True].copy()  # noqa: E712

    # Create birth year and month dummies for FE
    for year in BIRTH_COHORTS:
        analysis[f"byear_{year}"] = (analysis["birth_year"] == year).astype(int)

    for month in range(1, 13):
        analysis[f"bmonth_{month}"] = (analysis["birth_month"] == month).astype(int)

    return analysis

def get_sample_stats(sample: pd.DataFrame) -> dict:
    """Compute summary statistics for the analysis sample.

    Args:
        sample: Prepared analysis sample from prepare_analysis_sample().

    Returns:
        Dictionary with keys like n_obs, n_units, year_range, etc.
    """
    stats = {
        "n_obs": len(sample),
        "n_birthplaces": sample[PANEL_VAR].nunique(),
        "year_range": f"{sample['birth_year'].min()}-{sample['birth_year'].max()}",
        "veteran_rate": sample["veteran"].mean(),
        "draft_risk_rate": sample["draft_risk"].mean(),
        "compliance_rate": (
            sample[sample["draft_risk"] == 1]["veteran"].mean()
            - sample[sample["draft_risk"] == 0]["veteran"].mean()
        ),
    }
    return stats

In [ ]:
sample_west = prepare_analysis_sample(load_main_data(), sample="western")
print(f"Western sample shape: {sample_west.shape}")
stats = get_sample_stats(sample_west)
print(f"Observations: {stats['n_obs']}")
print(f"Veteran rate: {stats['veteran_rate']:.1%}")
print(f"Compliance rate: {stats['compliance_rate']:.1%}")
print(f"Top birthplaces:")
print(sample_west['bpl'].value_counts().head())


### Subsample: Non-Western Origin

Create non-Western subsample (Mexico, Latin America, Asia, Africa,
US territories). This subsample has ~15,000-18,500 observations and
is used for heterogeneity analysis in Table 4.

In [ ]:
def prepare_analysis_sample(df: pd.DataFrame, sample: str = "pooled") -> pd.DataFrame:
    """Apply sample restrictions and construct derived variables.

    Args:
        df: Raw loaded DataFrame from load_main_data().
        sample: Which sample to use - "pooled", "western", or "nonwestern"

    Returns:
        Analysis-ready DataFrame with all filters and transformations applied.
    """
    try:
        from .config import SAMPLES
    except ImportError:
        from config import SAMPLES

    # Select appropriate sample
    sample_var = SAMPLES[sample]
    analysis = df[df[sample_var] == True].copy()  # noqa: E712

    # Create birth year and month dummies for FE
    for year in BIRTH_COHORTS:
        analysis[f"byear_{year}"] = (analysis["birth_year"] == year).astype(int)

    for month in range(1, 13):
        analysis[f"bmonth_{month}"] = (analysis["birth_month"] == month).astype(int)

    return analysis

def get_sample_stats(sample: pd.DataFrame) -> dict:
    """Compute summary statistics for the analysis sample.

    Args:
        sample: Prepared analysis sample from prepare_analysis_sample().

    Returns:
        Dictionary with keys like n_obs, n_units, year_range, etc.
    """
    stats = {
        "n_obs": len(sample),
        "n_birthplaces": sample[PANEL_VAR].nunique(),
        "year_range": f"{sample['birth_year'].min()}-{sample['birth_year'].max()}",
        "veteran_rate": sample["veteran"].mean(),
        "draft_risk_rate": sample["draft_risk"].mean(),
        "compliance_rate": (
            sample[sample["draft_risk"] == 1]["veteran"].mean()
            - sample[sample["draft_risk"] == 0]["veteran"].mean()
        ),
    }
    return stats

In [ ]:
sample_nonwest = prepare_analysis_sample(load_main_data(), sample="nonwestern")
print(f"Non-Western sample shape: {sample_nonwest.shape}")
stats = get_sample_stats(sample_nonwest)
print(f"Observations: {stats['n_obs']}")
print(f"Veteran rate: {stats['veteran_rate']:.1%}")
print(f"Compliance rate: {stats['compliance_rate']:.1%}")
print(f"Top birthplaces:")
print(sample_nonwest['bpl'].value_counts().head())


## Descriptive Statistics


### Descriptive Statistics

Summary statistics for pooled, Western-origin, and Non-Western-origin immigrant samples (Table A4.1)

In [ ]:
def run(sample: pd.DataFrame = None, sample_name: str = "pooled") -> dict:
    """Compute descriptive statistics for the analysis sample.

    Args:
        sample: Analysis sample DataFrame (if None, will load from data module)
        sample_name: Sample identifier for output files

    Returns:
        Dictionary with summary statistics
    """
    if sample is None:
        from .data import load_main_data, prepare_analysis_sample

        df = load_main_data()
        sample = prepare_analysis_sample(df, sample=sample_name)

    print(f"\n{'='*60}")
    print(f"Descriptive Statistics: {sample_name.upper()}")
    print(f"{'='*60}\n")

    # Basic sample characteristics
    print(f"Total observations: {len(sample):,}")
    print(f"Birth years: {sample['birth_year'].min()}-{sample['birth_year'].max()}")
    print(f"Unique birthplaces: {sample['bpl'].nunique()}")
    print(f"Veteran rate: {sample[TREATMENT_VAR].mean():.3f}")
    print(f"Draft risk rate: {sample['draft_risk'].mean():.3f}")

    compliance = (
        sample[sample["draft_risk"] == 1][TREATMENT_VAR].mean()
        - sample[sample["draft_risk"] == 0][TREATMENT_VAR].mean()
    )
    print(f"Compliance rate: {compliance:.3f}\n")

    # Variables to summarize
    key_vars = [
        TREATMENT_VAR,
        "draft_risk",
        "age_immig",
        "yrs_since_immig",
    ] + PRIMARY_OUTCOMES + SECONDARY_OUTCOMES

    # Filter to variables that exist
    key_vars = [v for v in key_vars if v in sample.columns]

    # Compute summary statistics
    print("Computing summary statistics...")
    stats = compute_summary_stats(sample, key_vars, weights="pwt")

    print("\n" + "="*80)
    print(stats.to_string(index=False))
    print("="*80)

    # Save to CSV and LaTeX
    csv_file = f"descriptives_{sample_name}.csv"
    stats.to_csv(TABLE_DIR / csv_file, index=False)
    print(f"\nSaved: {TABLE_DIR / csv_file}")

    # Create LaTeX table
    latex = stats.to_latex(
        index=False,
        float_format="%.3f",
        caption=f"Descriptive Statistics: {sample_name.title()} Sample",
        label=f"tab:desc_{sample_name}",
    )
    tex_file = f"descriptives_{sample_name}.tex"
    save_table(latex, tex_file, TABLE_DIR)

    # Cross-tabulation: Veteran status by draft risk
    print("\n" + "="*60)
    print("CROSS-TABULATION: Veteran Status by Draft Risk")
    print("="*60)

    crosstab = pd.crosstab(
        sample["draft_risk"],
        sample[TREATMENT_VAR],
        normalize="index",
        margins=True,
    )
    print(crosstab)

    crosstab_counts = pd.crosstab(
        sample["draft_risk"],
        sample[TREATMENT_VAR],
        margins=True,
    )
    print("\nCounts:")
    print(crosstab_counts)

    # Save crosstab
    crosstab_file = f"crosstab_veteran_{sample_name}.csv"
    crosstab_counts.to_csv(TABLE_DIR / crosstab_file)
    print(f"\nSaved: {TABLE_DIR / crosstab_file}")

    return {
        "stats": stats,
        "crosstab": crosstab,
        "n_obs": len(sample),
        "compliance_rate": compliance,
    }

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## First-Stage Analysis


### First-Stage Analysis

Instrument relevance: draft lottery → veteran status. Estimates compliance rates and weak instrument tests (Table 1, Table A6.2)

In [ ]:
def run(sample: pd.DataFrame = None, sample_name: str = "pooled") -> dict:
    """Run first-stage analysis.

    Args:
        sample: Analysis sample DataFrame (if None, will load from data module)
        sample_name: Sample identifier for output files

    Returns:
        Dictionary with results
    """
    if sample is None:
        from .data import load_main_data, prepare_analysis_sample

        df = load_main_data()
        sample = prepare_analysis_sample(df, sample=sample_name)

    print(f"\n{'='*60}")
    print(f"First Stage Analysis: {sample_name.upper()}")
    print(f"{'='*60}\n")

    # Build control list (age_immig + birth year/month dummies)
    controls = CONTROLS.copy()
    for year in BIRTH_COHORTS:
        controls.append(f"byear_{year}")
    for month in range(1, 13):
        controls.append(f"bmonth_{month}")

    # Remove one category from each set of dummies to avoid collinearity
    controls.remove("byear_1949")
    controls.remove("bmonth_1")

    # Run first stage
    print(f"Dependent variable: {TREATMENT_VAR}")
    print(f"Instrument: {INSTRUMENT_VAR}")
    print("Controls: age_immig + birth year FE + birth month FE")
    print(f"Entity effects: {PANEL_VAR} (birthplace)")
    print(f"Sample size: {len(sample):,}\n")

    result = first_stage_diagnostics(
        data=sample,
        treatment=TREATMENT_VAR,
        instrument=INSTRUMENT_VAR,
        controls=controls,
        entity_effects=PANEL_VAR,
    )

    # Extract results
    model = result["model"]
    f_stat = result["f_statistic"]
    coef = result["coef"]

    print(f"First-stage coefficient on {INSTRUMENT_VAR}: {coef:.4f}")
    if hasattr(model, "std_errors"):
        se = model.std_errors.get(INSTRUMENT_VAR, None)
        if se:
            print(f"Standard error: {se:.4f}")
            print(f"t-statistic: {coef/se:.2f}")

    if f_stat:
        print(f"F-statistic: {f_stat:.2f}")

    # Compute compliance rate (reduced form)
    compliance = (
        sample[sample[INSTRUMENT_VAR] == 1][TREATMENT_VAR].mean()
        - sample[sample[INSTRUMENT_VAR] == 0][TREATMENT_VAR].mean()
    )
    print(f"Compliance rate (LATE denominator): {compliance:.4f}")

    # Save results to CSV
    results_df = pd.DataFrame(
        {
            "Sample": [sample_name],
            "N": [len(sample)],
            "Draft_Risk_Coef": [coef],
            "F_Statistic": [f_stat if f_stat else "N/A"],
            "Compliance_Rate": [compliance],
        }
    )

    output_file = f"first_stage_{sample_name}.csv"
    results_df.to_csv(TABLE_DIR / output_file, index=False)
    print(f"\nSaved: {TABLE_DIR / output_file}")

    return {
        "model": model,
        "coefficient": coef,
        "f_statistic": f_stat,
        "compliance_rate": compliance,
        "n_obs": len(sample),
    }

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Main IV Analysis


### Main IV Analysis

2SLS estimates of military service → integration outcomes. OLS, ITT, and IV specifications for 7 integration measures (Tables 2-4)

In [ ]:
def run(sample: pd.DataFrame = None, sample_name: str = "pooled") -> dict:
    """Run main IV analysis for all integration outcomes.

    Args:
        sample: Analysis sample DataFrame (if None, will load from data module)
        sample_name: Sample identifier for output files

    Returns:
        Dictionary with results for each outcome
    """
    if sample is None:
        from .data import load_main_data, prepare_analysis_sample

        df = load_main_data()
        sample = prepare_analysis_sample(df, sample=sample_name)

    print(f"\n{'='*60}")
    print(f"Main IV Analysis: {sample_name.upper()}")
    print(f"{'='*60}\n")

    # Build control list
    controls = CONTROLS.copy()
    for year in BIRTH_COHORTS:
        controls.append(f"byear_{year}")
    for month in range(1, 13):
        controls.append(f"bmonth_{month}")

    # Remove reference categories
    controls.remove("byear_1949")
    controls.remove("bmonth_1")

    # Store results
    results = {}

    # For each outcome, run OLS, ITT (reduced form), and 2SLS
    for outcome in PRIMARY_OUTCOMES:
        print(f"\nOutcome: {outcome}")
        print("-" * 40)

        # Check if outcome exists and has variation
        if outcome not in sample.columns:
            print(f"  WARNING: {outcome} not in data, skipping")
            continue

        outcome_data = sample[outcome].dropna()
        if len(outcome_data) < 100:
            print(f"  WARNING: Too few observations ({len(outcome_data)}), skipping")
            continue

        # (1) Naive OLS: outcome ~ veteran + controls + bpl FE
        print("  Running OLS...")
        try:
            ols_result = run_ols(sample, outcome, TREATMENT_VAR, controls, PANEL_VAR)
            ols_coef = ols_result.params.get(TREATMENT_VAR, None)
            ols_se = ols_result.std_errors.get(TREATMENT_VAR, None) if hasattr(
                ols_result, "std_errors"
            ) else None
            print(f"    OLS coef: {ols_coef:.4f} ({ols_se:.4f})")
        except Exception as e:
            print(f"    OLS failed: {e}")
            ols_result = None
            ols_coef = None
            ols_se = None

        # (2) ITT (reduced form): outcome ~ draft_risk + controls + bpl FE
        print("  Running ITT (reduced form)...")
        try:
            itt_result = run_ols(sample, outcome, INSTRUMENT_VAR, controls, PANEL_VAR)
            itt_coef = itt_result.params.get(INSTRUMENT_VAR, None)
            itt_se = itt_result.std_errors.get(INSTRUMENT_VAR, None) if hasattr(
                itt_result, "std_errors"
            ) else None
            print(f"    ITT coef: {itt_coef:.4f} ({itt_se:.4f})")
        except Exception as e:
            print(f"    ITT failed: {e}")
            itt_result = None
            itt_coef = None
            itt_se = None

        # (3) 2SLS: outcome ~ veteran (instrumented) + controls + bpl FE
        print("  Running 2SLS...")
        try:
            iv_result = run_2sls(
                sample, outcome, TREATMENT_VAR, INSTRUMENT_VAR, controls, PANEL_VAR
            )
            iv_coef = iv_result.params.get(TREATMENT_VAR, None)
            iv_se = iv_result.std_errors.get(TREATMENT_VAR, None)
            print(f"    2SLS coef: {iv_coef:.4f} ({iv_se:.4f})")
        except Exception as e:
            print(f"    2SLS failed: {e}")
            iv_result = None
            iv_coef = None
            iv_se = None

        # Store results
        results[outcome] = {
            "ols_model": ols_result,
            "ols_coef": ols_coef,
            "ols_se": ols_se,
            "itt_model": itt_result,
            "itt_coef": itt_coef,
            "itt_se": itt_se,
            "iv_model": iv_result,
            "iv_coef": iv_coef,
            "iv_se": iv_se,
        }

    # Create summary table
    print("\n" + "="*60)
    print("SUMMARY TABLE")
    print("="*60)

    summary_rows = []
    for outcome, res in results.items():
        row = {
            "Outcome": outcome,
            "OLS_Coef": res["ols_coef"],
            "OLS_SE": res["ols_se"],
            "ITT_Coef": res["itt_coef"],
            "ITT_SE": res["itt_se"],
            "IV_Coef": res["iv_coef"],
            "IV_SE": res["iv_se"],
        }
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)
    print(summary_df.to_string(index=False))

    # Save to CSV
    output_file = f"main_iv_{sample_name}.csv"
    summary_df.to_csv(TABLE_DIR / output_file, index=False)
    print(f"\nSaved: {TABLE_DIR / output_file}")

    return results

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Robustness Checks


### Robustness Checks

Attrition analysis, cohort-specific estimates, and exclusion restriction tests (Tables A8.4-A11.7)

In [ ]:
def run(sample: pd.DataFrame = None, sample_name: str = "pooled") -> dict:
    """Run robustness checks.

    Args:
        sample: Analysis sample DataFrame (if None, will load from data module)
        sample_name: Sample identifier for output files

    Returns:
        Dictionary with robustness test results
    """
    if sample is None:
        from .data import load_main_data, prepare_analysis_sample

        df = load_main_data()
        sample = prepare_analysis_sample(df, sample=sample_name)

    print(f"\n{'='*60}")
    print(f"Robustness Checks: {sample_name.upper()}")
    print(f"{'='*60}\n")

    results = {}

    # -------------------------------------------------------------------------
    # 1. Excess mortality test (Table A8.4)
    # -------------------------------------------------------------------------
    print("\n" + "-"*60)
    print("1. EXCESS MORTALITY TEST")
    print("-"*60)
    print("Testing whether veterans have higher mortality than expected.")
    print("(In Census 2000, measuring survival to 2000 for Vietnam-era cohorts)\n")

    # For demonstration: assume we can calculate expected survival
    # In reality, this requires external mortality tables
    veteran_rate = sample[TREATMENT_VAR].mean()
    expected_rate = 0.16  # Hypothetical baseline from non-combat data

    n_obs = len(sample)
    mortality_test = proportion_test(veteran_rate, expected_rate, n_obs)

    print(f"Observed veteran rate: {mortality_test['observed']:.4f}")
    print(f"Expected rate (if no excess mortality): {mortality_test['expected']:.4f}")
    print(f"Z-statistic: {mortality_test['z_statistic']:.3f}")
    print(f"P-value: {mortality_test['p_value']:.4f}")

    if mortality_test['p_value'] > 0.05:
        print("=> No significant excess mortality detected")
    else:
        print("=> Significant difference detected (possible attrition bias)")

    results["mortality_test"] = mortality_test

    # -------------------------------------------------------------------------
    # 2. Attrition by nationality (Table A8.5)
    # -------------------------------------------------------------------------
    print("\n" + "-"*60)
    print("2. ATTRITION BY NATIONALITY")
    print("-"*60)
    print("Testing for differential attrition by birthplace.\n")

    # For each major birthplace, test if veteran rate differs from pooled
    top_birthplaces = sample["bpl"].value_counts().head(10).index

    attrition_tests = []
    for bpl_code in top_birthplaces:
        bpl_sample = sample[sample["bpl"] == bpl_code]
        n_bpl = len(bpl_sample)

        if n_bpl < 50:
            continue

        bpl_vet_rate = bpl_sample[TREATMENT_VAR].mean()
        test = proportion_test(bpl_vet_rate, veteran_rate, n_bpl)

        attrition_tests.append(
            {
                "Birthplace": int(bpl_code),
                "N": n_bpl,
                "Vet_Rate": bpl_vet_rate,
                "Z_Stat": test["z_statistic"],
                "P_Value": test["p_value"],
            }
        )

    attrition_df = pd.DataFrame(attrition_tests)
    print(attrition_df.to_string(index=False))

    # Save
    attrition_file = f"attrition_by_nationality_{sample_name}.csv"
    attrition_df.to_csv(TABLE_DIR / attrition_file, index=False)
    print(f"\nSaved: {TABLE_DIR / attrition_file}")

    results["attrition_tests"] = attrition_df

    # -------------------------------------------------------------------------
    # 3. Exclusion restriction: Subgroup analysis (Table A10.6)
    # -------------------------------------------------------------------------
    print("\n" + "-"*60)
    print("3. EXCLUSION RESTRICTION TEST")
    print("-"*60)
    print("Testing whether draft risk affects outcomes only through service.\n")
    print("Strategy: Estimate effect separately for veterans and non-veterans.")
    print("If exclusion holds, draft_risk should have no effect among non-veterans.\n")

    # Build controls
    controls = CONTROLS.copy()
    for year in BIRTH_COHORTS:
        controls.append(f"byear_{year}")
    for month in range(1, 13):
        controls.append(f"bmonth_{month}")
    controls.remove("byear_1949")
    controls.remove("bmonth_1")

    # Test on one outcome: naturalization
    test_outcome = "naturalized"

    if test_outcome in sample.columns:
        # Split by veteran status
        veterans = sample[sample[TREATMENT_VAR] == 1]
        non_veterans = sample[sample[TREATMENT_VAR] == 0]

        print(f"Testing outcome: {test_outcome}")
        print(f"Veterans: N={len(veterans):,}")
        print(f"Non-veterans: N={len(non_veterans):,}\n")

        exclusion_results = {}

        # For each subgroup, regress outcome on draft_risk (reduced form)
        from .helpers import run_ols

        for group_name, group_data in [("Veterans", veterans), ("Non-Veterans", non_veterans)]:
            if len(group_data) < 100:
                print(f"{group_name}: Too few observations, skipping")
                continue

            try:
                model = run_ols(group_data, test_outcome, INSTRUMENT_VAR, controls, PANEL_VAR)
                coef = model.params.get(INSTRUMENT_VAR, None)
                se = model.std_errors.get(INSTRUMENT_VAR, None) if hasattr(
                    model, "std_errors"
                ) else None

                print(f"{group_name}:")
                print(f"  Draft risk coef: {coef:.4f} (SE: {se:.4f})")
                print(f"  T-stat: {coef/se:.2f}" if se else "")

                exclusion_results[group_name] = {
                    "coef": coef,
                    "se": se,
                    "n_obs": len(group_data),
                }
            except Exception as e:
                print(f"{group_name}: Failed - {e}")

        # Interpretation
        if "Non-Veterans" in exclusion_results:
            nv_coef = exclusion_results["Non-Veterans"]["coef"]
            nv_se = exclusion_results["Non-Veterans"]["se"]
            if abs(nv_coef / nv_se) < 1.96:
                print("\n=> Exclusion restriction holds: No effect among non-veterans")
            else:
                print(
                    "\n=> WARNING: Draft risk affects outcome among non-veterans "
                    "(exclusion restriction violated)"
                )

        results["exclusion_test"] = exclusion_results

    return results

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))